# Prévision de la consommation électrique (Stacked LSTM multivarié)
## Prédiction de la puissance active globale avec un LSTM empilé à 3 couches

**Objectif du TP :**
- Prédire la puissance active globale (`Global_active_power`) d'un foyer
- Utiliser un **LSTM empilé à 3 couches** (architecture multivarié)
- Ré-échantillonner les données à l'heure (le fichier brut contient ~2 millions de lignes à la minute)
- Ajouter un **encodage cyclique de l'heure** (`hour_sin`, `hour_cos`) pour 10 variables au total
- Normaliser uniquement sur l'ensemble d'entraînement (évite la fuite d'information)

Jeu de données : *Individual Household Electric Power Consumption* (UCI Machine Learning Repository).

## 0. Imports et configuration

In [ ]:
import os, zipfile, urllib.request, time

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt



import torch

import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader



from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



import warnings

warnings.filterwarnings("ignore")



# ─── Dossier de sauvegarde des figures ────────────────────────────────────────

os.makedirs("figs", exist_ok=True)



# ─── Appareil de calcul ───────────────────────────────────────────────────────

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Appareil utilisé : {device}")



# ─── Hyperparamètres globaux ──────────────────────────────────────────────────

SEQ_LENGTH = 24      # fenêtre d'observation : 24 heures pour prédire l'heure suivante

HIDDEN     = 128     # taille des états cachés du LSTM

NUM_LAYERS = 3       # nombre de couches LSTM empilées

DROPOUT    = 0.3     # taux de dropout entre les couches LSTM

EPOCHS     = 30      # nombre d'époques d'entraînement

BATCH      = 64      # taille des mini-batches

LR         = 1e-3    # taux d'apprentissage (Adam)

SEED       = 42      # graine aléatoire pour la reproductibilité



np.random.seed(SEED)

torch.manual_seed(SEED)

print("✓ Configuration initialisée")

## 1. Chargement des données

### Présentation du jeu de données

| Propriété | Valeur |
|-----------|--------|
| Source | UCI Machine Learning Repository |
| Période | Décembre 2006 → Novembre 2010 |
| Granularité | 1 minute (~2 millions de lignes) |
| Cible | `Global_active_power` (puissance active en kW) |
| Valeurs manquantes | notées `?` |

Téléchargement automatique depuis l'UCI. En cas d'échec, déposer manuellement `household_power_consumption.txt` dans le répertoire de travail.

In [ ]:
# ─── Téléchargement automatique du dataset UCI ────────────────────────────────

TXT = "household_power_consumption.txt"

UCI_URL = ("https://archive.ics.uci.edu/static/public/235/"

           "individual+household+electric+power+consumption.zip")



if not os.path.exists(TXT):

    try:

        print("Téléchargement depuis UCI ...")

        urllib.request.urlretrieve(UCI_URL, "hpc.zip")

        with zipfile.ZipFile("hpc.zip") as z:

            z.extractall(".")

        # le zip UCI contient parfois un second zip imbriqué

        if not os.path.exists(TXT):

            for f in os.listdir("."):

                if f.endswith(".zip") and f != "hpc.zip":

                    with zipfile.ZipFile(f) as z:

                        z.extractall(".")

        print(f"✓ Fichier récupéré : {TXT}")

    except Exception as e:

        print("Échec du téléchargement :", e)

        print("→ Déposez manuellement household_power_consumption.txt puis relancez.")



# ─── Lecture du fichier (séparateur ';', valeurs manquantes '?') ──────────────

data = pd.read_csv(TXT, sep=";", na_values=["?"], low_memory=False,

                   parse_dates={"datetime": ["Date", "Time"]},

                   dayfirst=True)

data = data.dropna().reset_index(drop=True)

data = data.set_index("datetime").sort_index()



print(f"Lignes (minute) après nettoyage : {len(data):,}")

print(f"Période : {data.index.min()}  →  {data.index.max()}")

## 2. Ré-échantillonnage horaire et feature engineering

### Pourquoi ré-échantillonner ?

Le fichier brut contient ~2 millions de lignes à la minute. Créer des séquences
de longueur 24 directement dessus saturerait la RAM et ralentirait inutilement
l'entraînement. On agrège donc à l'heure (moyenne).

### Encodage cyclique de l'heure

L'heure (0–23) est une variable **cyclique** : l'heure 23 est proche de l'heure 0.
Un encodage linéaire brut ne capture pas cette continuité. La solution :

```
hour_sin = sin(2π × heure / 24)
hour_cos = cos(2π × heure / 24)
```

Ces deux coordonnées projettent l'heure sur un cercle unité, préservant la
continuité 23→0.

In [ ]:
# ─── Conversion des colonnes numériques ───────────────────────────────────────

num_cols = ["Global_active_power", "Global_reactive_power", "Voltage",

            "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]

data[num_cols] = data[num_cols].astype(float)



# ─── Agrégation horaire (moyenne) ─────────────────────────────────────────────

df = data[num_cols].resample("1h").mean().dropna()



# ─── Feature engineering : variables temporelles ──────────────────────────────

df["hour"]       = df.index.hour                          # heure brute (0–23)

df["dayofweek"]  = df.index.dayofweek                     # jour de la semaine (0=lundi)

df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)    # 1 si week-end, 0 sinon

df["hour_sin"]   = np.sin(2 * np.pi * df["hour"] / 24)   # composante sinus de l'heure

df["hour_cos"]   = np.cos(2 * np.pi * df["hour"] / 24)   # composante cosinus de l'heure



print(f"Lignes (horaire) : {len(df):,}")

print(f"Variables : {list(df.columns)}")

df.head()

In [ ]:
# ─── Visualisation : profil de consommation sur 2 semaines ────────────────────

plt.figure(figsize=(14, 4))

df["Global_active_power"].iloc[:24 * 14].plot()

plt.title("Consommation (Global_active_power) — 2 premières semaines")

plt.xlabel("Date")

plt.ylabel("Puissance active (kW)")

plt.grid(True)

plt.tight_layout()

plt.savefig("figs/tp2_overview.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp2_overview.png")

## 3. Sélection des variables, normalisation et création des séquences

### Variables utilisées (10 au total)

| Variable | Rôle |
|----------|------|
| `Global_active_power` | **Cible** à prédire (colonne 0) |
| `Global_reactive_power` | Puissance réactive |
| `Voltage` | Tension du réseau |
| `Global_intensity` | Intensité globale |
| `Sub_metering_1/2/3` | Sous-compteurs (cuisine, salle de bain, autres) |
| `hour_sin`, `hour_cos` | Encodage cyclique de l'heure |
| `is_weekend` | Indicateur week-end |

### Normalisation sans fuite d'information

Le `MinMaxScaler` est **ajusté uniquement sur l'ensemble d'entraînement**.
L'appliquer sur l'intégralité des données ferait fuir des informations du
futur vers le passé (data leakage).

In [ ]:
# ─── Sélection des 10 variables ───────────────────────────────────────────────

feature_cols = ["Global_active_power", "Global_reactive_power", "Voltage",

                "Global_intensity", "Sub_metering_1", "Sub_metering_2",

                "Sub_metering_3", "hour_sin", "hour_cos", "is_weekend"]

dataset = df[feature_cols].values.astype(np.float32)

print("Forme du dataset :", dataset.shape, "(colonne 0 = cible)")



# ─── Split train/test au niveau des lignes brutes ─────────────────────────────

#     (avant la création des séquences, pour un ajustement propre du scaler)

n          = len(dataset)

train_rows = int(n * 0.8)   # 80 % entraînement, 20 % test



# ─── Normalisation MinMax ajustée sur l'entraînement uniquement ───────────────

feat_scaler   = MinMaxScaler().fit(dataset[:train_rows])

scaled        = feat_scaler.transform(dataset).astype(np.float32)



# scaler dédié à la cible — pour revenir à l'échelle réelle (kW) après prédiction

target_scaler = MinMaxScaler().fit(dataset[:train_rows, [0]])



# ─── Création des séquences glissantes ────────────────────────────────────────

def create_sequences(arr, seq_len):

    """Découpe arr en fenêtres de longueur seq_len ; la cible est le pas suivant."""

    X, y = [], []

    for i in range(len(arr) - seq_len):

        X.append(arr[i:i + seq_len])       # fenêtre d'entrée

        y.append(arr[i + seq_len, 0])      # valeur suivante de Global_active_power

    return np.asarray(X, np.float32), np.asarray(y, np.float32)



X, y = create_sequences(scaled, SEQ_LENGTH)

print(f"Séquences créées : X {X.shape}  |  y {y.shape}")



# ─── Split chronologique des séquences ────────────────────────────────────────

#     On ne mélange PAS (shuffle=False) pour respecter l'ordre temporel

split   = int(len(X) * 0.8)

X_train = torch.from_numpy(X[:split])

X_test  = torch.from_numpy(X[split:])

y_train = torch.from_numpy(y[:split]).unsqueeze(-1)

y_test  = torch.from_numpy(y[split:]).unsqueeze(-1)

print(f"Train : {X_train.shape[0]:,} séquences  |  Test : {X_test.shape[0]:,} séquences")



# ─── DataLoaders ──────────────────────────────────────────────────────────────

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=BATCH, shuffle=True)

test_loader  = DataLoader(TensorDataset(X_test,  y_test),  batch_size=BATCH, shuffle=False)

print("✓ DataLoaders prêts")

## 4. Architecture : Stacked LSTM à 3 couches

### Pourquoi empiler plusieurs couches LSTM ?

Un LSTM à couche unique capture des dépendances temporelles de bas niveau.
En empilant 3 couches, on crée une **hiérarchie de représentations** :

```
Entrée (24 pas × 10 variables)
       ↓
 [LSTM couche 1 — hidden=128]   ← motifs locaux (heures)
       ↓
 [LSTM couche 2 — hidden=128]   ← motifs intermédiaires (journées)
       ↓
 [LSTM couche 3 — hidden=128]   ← tendances de plus haut niveau
       ↓
 [Linear(128 → 1)]              ← prédiction scalaire (kW)
```

Le **Dropout** (30 %) entre les couches limite le sur-apprentissage.

In [ ]:
# ─── Définition du Stacked LSTM ───────────────────────────────────────────────

class StackedLSTM(nn.Module):

    def __init__(self, input_size, hidden_size=128, num_layers=3, dropout=0.3):

        """

        LSTM empilé pour la prédiction de séries temporelles multivariées.



        Paramètres :

            input_size  : nombre de variables d'entrée (10 ici)

            hidden_size : dimension des états cachés

            num_layers  : nombre de couches LSTM empilées

            dropout     : taux de dropout entre les couches (sauf la dernière)

        """

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=input_size,

            hidden_size=hidden_size,

            num_layers=num_layers,

            batch_first=True,    # (batch, seq, features) — plus intuitif

            dropout=dropout

        )

        self.fc = nn.Linear(hidden_size, 1)   # projection finale → scalaire



    def forward(self, x):

        """Passage avant : on ne garde que la dernière sortie temporelle."""

        out, _ = self.lstm(x)          # out : (batch, seq_len, hidden_size)

        return self.fc(out[:, -1, :])  # dernière sortie → (batch, 1)



# ─── Instanciation du modèle ──────────────────────────────────────────────────

model = StackedLSTM(

    input_size=X.shape[2],

    hidden_size=HIDDEN,

    num_layers=NUM_LAYERS,

    dropout=DROPOUT

).to(device)



print(model)

total_params = sum(p.numel() for p in model.parameters())

print(f"\n✓ Paramètres totaux : {total_params:,}")

## 5. Entraînement

Perte **MSE**, optimiseur **Adam**. On suit la perte d'entraînement et de test à chaque époque.

In [ ]:
# ─── Fonction et optimiseur de perte ──────────────────────────────────────────

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)



def run_epoch(loader, train=True):

    """Exécute une époque complète (entraînement ou évaluation)."""

    model.train() if train else model.eval()

    total, n = 0.0, 0

    with torch.set_grad_enabled(train):

        for Xb, yb in loader:

            Xb, yb = Xb.to(device), yb.to(device)

            if train:

                optimizer.zero_grad()

            out  = model(Xb)

            loss = criterion(out, yb)

            if train:

                loss.backward()

                optimizer.step()

            total += loss.item() * Xb.size(0)

            n     += Xb.size(0)

    return total / n



# ─── Boucle d'entraînement ────────────────────────────────────────────────────

hist = {"train": [], "test": []}

t0   = time.time()



for epoch in range(EPOCHS):

    tr = run_epoch(train_loader, train=True)

    te = run_epoch(test_loader,  train=False)

    hist["train"].append(tr)

    hist["test"].append(te)

    if (epoch + 1) % 5 == 0 or epoch == 0:

        print(f"Epoch {epoch+1:3d}/{EPOCHS} | Train MSE : {tr:.5f} | Test MSE : {te:.5f}")



print(f"\n✓ Entraînement terminé en {time.time()-t0:.1f}s")

In [ ]:
# ─── Courbes de perte Train / Test ────────────────────────────────────────────

plt.figure(figsize=(9, 5))

plt.plot(hist["train"], label="Train")

plt.plot(hist["test"],  label="Test")

plt.title("Stacked LSTM — Courbe de perte MSE")

plt.xlabel("Époque")

plt.ylabel("MSE")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig("figs/tp2_loss.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp2_loss.png")

## 6. Prédictions et retour à l'échelle réelle

Le modèle prédit des valeurs normalisées (MinMax). On utilise `target_scaler.inverse_transform` pour revenir aux kW réels.

In [ ]:
# ─── Génération des prédictions ───────────────────────────────────────────────

model.eval()

with torch.no_grad():

    y_pred_scaled = model(X_test.to(device)).cpu().numpy()



# ─── Dénormalisation (retour en kW) ───────────────────────────────────────────

y_test_real = target_scaler.inverse_transform(y_test.numpy().reshape(-1, 1))

y_pred_real = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))



# ─── Visualisation : réel vs prédit (500 premiers points de test) ─────────────

plt.figure(figsize=(15, 6))

plt.plot(y_test_real[:500], label="Consommation réelle",  color="blue")

plt.plot(y_pred_real[:500], label="Prédiction LSTM",      color="red", alpha=0.8)

plt.title("Prévision de la consommation électrique — Stacked LSTM (3 couches)")

plt.xlabel("Heures (ensemble de test)")

plt.ylabel("Global_active_power (kW)")

plt.legend()

plt.grid(True)

plt.tight_layout()

plt.savefig("figs/tp2_pred.png", dpi=200, bbox_inches="tight")

plt.show()

print("✓ Figure sauvegardée : figs/tp2_pred.png")

## 7. Évaluation des performances

### Métriques utilisées

| Métrique | Description |
|----------|-------------|
| MAE | Erreur absolue moyenne (en kW) |
| RMSE | Racine de l'erreur quadratique moyenne |
| R² | Coefficient de détermination (1 = parfait) |
| MAPE | Erreur absolue moyenne en pourcentage |

In [ ]:
# ─── Calcul des métriques ─────────────────────────────────────────────────────

mae  = mean_absolute_error(y_test_real, y_pred_real)

rmse = np.sqrt(mean_squared_error(y_test_real, y_pred_real))

r2   = r2_score(y_test_real, y_pred_real)

mape = np.mean(

    np.abs((y_test_real - y_pred_real) / np.clip(np.abs(y_test_real), 1e-6, None))

) * 100



print("─" * 40)

print(f"  MAE  : {mae:.4f} kW")

print(f"  RMSE : {rmse:.4f} kW")

print(f"  R²   : {r2:.4f}")

print(f"  MAPE : {mape:.2f} %")

print("─" * 40)



# ─── Sauvegarde des résultats ─────────────────────────────────────────────────

pd.DataFrame({"MAE": [mae], "RMSE": [rmse], "R2": [r2], "MAPE_%": [mape]}).to_csv(

    "figs/tp2_results.csv", index=False

)

print("✓ Résultats sauvegardés : figs/tp2_results.csv")